> **제4회 Level 모의고사** — 교재 600~703쪽
> 문제 600~602쪽 / 정답·해설 680~703쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

# 작업형 제1유형

## 문제 1. (10점)

제공된 데이터(crime_data.csv)는 연도별 범죄 발생 및 검거 데이터입니다. 아래 사항을 수행하여 [제출 형식]에 맞춰 답안을 입력하시오.

1. 검거율은 검거건수(`arrests`) / 발생건수(`cases`)이다.
2. 각 연도에서 가장 높은 검거율을 기록한 범죄유형(`crime_type`)을 찾으시오.
3. 해당 범죄유형의 검거건수를 구한 후, 이 값들을 모두(연도별로) 합산하시오.

**제공 데이터 (crime_data.csv)**
| 컬럼명 | 설명 |
|---|---|
| year | 연도 |
| crime_type | 범죄유형 |
| cases | 발생건수 |
| arrests | 검거건수 |

**[제출 형식]**
- 정수로 작성

### 문제 풀이 방향

이 데이터는 한 행이 특정 연도의 특정 범죄유형에 대한 발생건수와 검거건수를 담고 있는 형태입니다. 즉 2015년 한 해에도 절도, 폭행, 사기 등 여러 범죄유형이 각각 다른 행으로 존재합니다. 문제가 요구하는 것은 연도마다 그 해에 검거율이 가장 높았던 범죄유형 하나를 뽑고, 그 범죄유형의 검거건수를 확인한 뒤, 9개 연도에서 뽑힌 9개의 검거건수를 모두 더하는 것입니다. 그러므로 먼저 검거율이라는 새 컬럼을 만들고, 연도라는 기준으로 그룹을 나눠서 그룹마다 검거율이 최고인 행을 찾은 다음, 그 행들의 검거건수만 모아서 합산하는 순서로 풀어보겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 검거율 컬럼 만들기**

In [1]:
import pandas as pd

df = pd.read_csv('crime_data.csv')
df['arrest_rate'] = df['arrests'] / df['cases']

print(df[['year', 'crime_type', 'cases', 'arrests', 'arrest_rate']].head())

   year crime_type  cases  arrests  arrest_rate
0  2015         절도   2975     1616     0.543193
1  2015         폭행   1517     1211     0.798286
2  2015         사기   1127      549     0.487134
3  2015         방화   2988     1768     0.591700
4  2015         횡령   1790     1043     0.582682


`arrests`와 `cases`는 각각 검거건수와 발생건수 컬럼입니다. 두 컬럼을 나누는 연산은 같은 행끼리 자동으로 짝지어 계산되므로, 25행이든 90행이든 모든 행에 대해 반복문 없이 한 번에 검거율이 계산됩니다. 이렇게 나눗셈처럼 이미 있는 두 컬럼으로부터 새로운 의미를 가진 컬럼을 만드는 것을 파생변수를 생성한다고 표현합니다.

**셀 2) 연도별로 검거율이 가장 높은 행 찾기**

In [2]:
idx = df.groupby('year')['arrest_rate'].idxmax()
print(idx)

year
2015     1
2016     9
2017    17
2018    19
2019    27
2020    31
2021    39
2022    43
2023    49
Name: arrest_rate, dtype: int64


이 셀이 이번 문제에서 가장 중요한 부분입니다. `groupby('year')`로 연도가 같은 행들을 하나의 묶음으로 나눕니다. 그리고 그 뒤에 `['arrest_rate'].idxmax()`를 붙이면, 각 연도 묶음 안에서 arrest_rate 값이 가장 큰 행의 원본 인덱스(위치)를 알려줍니다. 즉 결과로 나온 숫자 1, 9, 17, ...은 df에서 몇 번째 행인지를 나타내는 위치 번호입니다. 예를 들어 2015년 묶음 안에서는 인덱스 1번 행(폭행)이 검거율이 가장 높았고, 2016년 묶음 안에서는 인덱스 9번 행(방화)이 가장 높았다는 뜻입니다.

여기서 idxmax가 각 그룹 안에서만 최댓값을 찾는다는 점을 다시 한번 짚어보겠습니다. 만약 groupby 없이 df 전체에 대해 `df['arrest_rate'].idxmax()`를 했다면, 90개 행을 통틀어 검거율이 가장 높은 딱 하나의 행만 찾게 됩니다. 하지만 우리가 원하는 것은 연도 9개 각각에 대해 하나씩, 즉 9개의 결과입니다. `groupby()`를 먼저 적용해서 데이터를 9개의 작은 묶음으로 나눠두면, 그 뒤에 붙는 idxmax가 전체가 아니라 각 묶음 안에서만 동작하게 됩니다. 이렇게 그룹별로 나눈 뒤 같은 계산을 반복 적용하는 방식은, 반복문으로 연도를 하나씩 돌면서 그 안에서 다시 최댓값을 찾는 이중 반복문과 정확히 같은 결과를 만들어내지만 코드는 훨씬 짧습니다.

**셀 3) 찾은 행들만 모아서 확인하기**

In [3]:
top = df.loc[idx]
print(top[['year', 'crime_type', 'arrest_rate', 'arrests']])

    year crime_type  arrest_rate  arrests
1   2015         폭행     0.798286     1211
9   2016         방화     0.885045      793
17  2017        성범죄     0.838188      259
19  2018         폭행     0.882698      602
27  2019         방화     0.757085      187
31  2020         폭행     0.739326      329
39  2021         방화     0.842589     1927
43  2022         폭행     0.769759      896
49  2023         폭행     0.818182      288


`df.loc[idx]`는 셀 2에서 구한 인덱스 목록(1, 9, 17, ...)에 해당하는 행들만 모아서 보여줍니다. `.loc[]`는 지정한 인덱스 값에 해당하는 행을 그대로 가져오는 방법입니다. 그 결과 9개 연도 각각에서 검거율이 가장 높았던 범죄유형의 정보가 한 표에 모이게 됩니다. 이 표를 보면 어떤 해에는 폭행이, 어떤 해에는 방화나 성범죄가 그 해의 최고 검거율 범죄유형이었다는 것을 확인할 수 있습니다.

**셀 4) 검거건수를 모두 합산하기 (정답)**

In [4]:
answer = top['arrests'].sum()
print("정답:", answer)

정답: 6492


`top`에 남아있는 것은 9개 행뿐이므로, 이 중 `arrests` 컬럼만 골라서 `.sum()`으로 모두 더하면 문제에서 요구하는 최종 합계가 나옵니다.

> 💡 `groupby(기준)[대상컬럼].idxmax()`는 그룹마다 어떤 조건에서 최댓값을 기록한 행이 무엇인지 찾아야 하는 문제에서 자주 쓰이는 패턴입니다. 결과로 나온 인덱스를 `df.loc[]`에 넣으면 그 행들의 다른 컬럼 값도 함께 확인할 수 있습니다.

### 전체 코드

In [5]:
import pandas as pd

df = pd.read_csv('crime_data.csv')
df['arrest_rate'] = df['arrests'] / df['cases']

idx = df.groupby('year')['arrest_rate'].idxmax()
top = df.loc[idx]

answer = top['arrests'].sum()
print("정답:", answer)

정답: 6492


### 정리

이 문제의 핵심은 groupby와 idxmax를 함께 사용하면 그룹별로 조건에 맞는 행을 한 번에 찾을 수 있다는 것입니다. idxmax를 groupby 없이 단독으로 쓰면 데이터 전체에서 딱 하나의 답만 나오지만, groupby로 먼저 연도를 나눠두면 같은 idxmax가 연도마다 각각 적용되어 원하는 개수만큼의 결과가 나옵니다. 그룹별로 나눈 다음에 같은 계산을 적용한다는 흐름은 앞으로도 자주 마주치게 되는 패턴이므로, groupby가 데이터를 몇 개의 조각으로 나누는 역할을 한다는 것과, 그 뒤에 붙는 함수는 그 조각들 안에서 각각 독립적으로 동작한다는 것을 함께 기억해두면 좋습니다.

---

## 문제 2. (10점)

제공된 데이터(sms_data.csv)는 문자 메시지 내용과 스팸 여부를 담고 있습니다. 아래 수행 순서에 따라 데이터를 처리한 뒤, 구한 값을 [제출 형식]에 맞춰 답안을 입력하시오.

**[수행 순서]**
1. 각 메시지의 단어 개수를 띄어쓰기(공백) 기준으로 계산하시오.
2. 스팸(spam) 메시지의 평균 단어 개수와 정상(ham) 메시지의 평균 단어 개수를 각각 구하시오.
3. 두 평균의 차이의 절댓값을 구하시오.

**제공 데이터 (sms_data.csv)**
| 컬럼명 | 설명 |
|---|---|
| label | 메시지 구분 (spam: 스팸, ham: 정상) |
| text | 메시지 내용 |

**[제출 형식]**
- 반올림하여 소수점 셋째 자리까지 작성

### 문제 풀이 방향

이 문제는 텍스트 데이터를 다루지만 구조 자체는 앞서 다룬 그룹별 평균 문제와 크게 다르지 않습니다. 먼저 각 메시지의 텍스트를 숫자(단어 개수)로 바꾸는 전처리가 필요하고, 그 다음엔 label(spam/ham)을 기준으로 그룹을 나눠 평균을 비교하면 됩니다. 텍스트를 숫자로 바꾸는 방법과, 그렇게 만든 숫자 컬럼에 그룹별 평균을 적용하는 두 단계로 나눠서 보겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [6]:
import pandas as pd

df = pd.read_csv('sms_data.csv')
print(df.head())

  label                                               text
0   ham        Thanks for the update see you tomorrow sure
1  spam  Congratulations you have won a free prize clai...
2   ham        Let's catch up this weekend if you are free
3   ham      Hey are we still meeting for lunch today sure
4  spam  Congratulations you have won a free prize clai...


**셀 2) 공백 기준으로 단어 개수 세기**

In [7]:
df['words'] = df['text'].apply(lambda x: len(x.split(' ')))
print(df[['label', 'text', 'words']].head())

  label                                               text  words
0   ham        Thanks for the update see you tomorrow sure      8
1  spam  Congratulations you have won a free prize clai...     11
2   ham        Let's catch up this weekend if you are free      9
3   ham      Hey are we still meeting for lunch today sure      9
4  spam  Congratulations you have won a free prize clai...     12


`x.split(' ')`은 문자열 x를 공백을 기준으로 잘라서 여러 개의 조각으로 나눈 리스트를 만듭니다. 예를 들어 "I am happy"라는 문자열에 split(' ')을 적용하면 ['I', 'am', 'happy']라는 세 개짜리 리스트가 되고, 이 리스트의 길이(len)는 3이 되어 단어 개수와 같아집니다.

`.apply(lambda x: ...)`는 컬럼에 있는 값 하나하나에 대해 지정한 동작을 반복해서 적용해주는 함수입니다. `df['text']`에는 여러 개의 문자열이 들어있는데, apply를 사용하면 그 문자열 하나하나에 대해 "공백으로 나눠서 길이를 잰다"는 동작이 자동으로 반복 적용됩니다. `lambda x: len(x.split(' '))`는 "x라는 입력을 받아서 len(x.split(' '))이라는 결과를 돌려주는 간단한 함수"를 그 자리에서 즉석으로 정의한 것입니다. def로 함수를 미리 만들어두고 이름을 붙여 호출하는 대신, 한 줄로 짧게 쓰기 위해 이런 익명 함수(lambda)를 사용했습니다.

> 💡 apply와 lambda의 조합은 pandas 내장 함수로 바로 처리되지 않는, 조금 더 개별적인 로직(문자열 자르기, 조건에 따른 값 변환 등)을 컬럼 전체에 적용하고 싶을 때 폭넓게 쓰이는 방법입니다. "컬럼의 각 값에 대해 이런 처리를 하고 싶다"는 문장이 떠오르면 `.apply(lambda x: ...)`를 떠올리면 됩니다.


**셀 3) label별 평균 단어 개수 구하기**

In [8]:
group_mean = df.groupby('label')['words'].mean()
print(group_mean)

label
ham      9.313333
spam    10.893333
Name: words, dtype: float64


`groupby('label')`로 spam과 ham을 각각 하나의 그룹으로 나누고, `['words'].mean()`으로 그룹 안에서 단어 개수의 평균을 구했습니다. 이 결과를 보면 스팸 메시지가 평균적으로 정상 메시지보다 단어 수가 조금 더 많다는 것을 알 수 있습니다.

**셀 4) 두 평균의 차이(절댓값) 구하기 (정답)**

In [9]:
answer = round(abs(group_mean['spam'] - group_mean['ham']), 3)
print("정답:", answer)

정답: 1.58


`group_mean`은 label을 인덱스로 가지는 Series이므로, `group_mean['spam']`과 `group_mean['ham']`처럼 인덱스 이름으로 각 그룹의 평균값을 바로 꺼낼 수 있습니다. 두 값을 뺀 뒤 `abs()`로 절댓값을 취해서, 어느 쪽이 더 큰지와 무관하게 차이의 크기만 구했습니다. 문제에서 요구한 것은 단순히 차이의 크기이므로, 부호는 중요하지 않고 절댓값만 있으면 충분합니다.

### 전체 코드

In [10]:
import pandas as pd

df = pd.read_csv('sms_data.csv')
df['words'] = df['text'].apply(lambda x: len(x.split(' ')))

group_mean = df.groupby('label')['words'].mean()
answer = round(abs(group_mean['spam'] - group_mean['ham']), 3)

print("정답:", answer)

정답: 1.58


### 정리

이 문제는 텍스트라는 조금 낯선 재료를 다뤘지만, 결국 핵심은 텍스트를 숫자(단어 개수)로 바꾸는 전처리 한 단계를 거친 뒤에는 지금까지 계속 봐온 groupby().mean() 패턴으로 그대로 이어진다는 것입니다. 데이터의 종류가 숫자든 텍스트든, 일단 비교하고 싶은 대상을 숫자 컬럼으로 만들어두면 그 다음 분석 흐름은 동일하게 적용된다는 점을 기억해두면 좋습니다.

---

## 문제 3. (10점) — 신규 유형

제공된 데이터(customer_age.csv)는 고객의 나이와 소비 금액 데이터입니다. 아래 수행 순서에 따라 데이터를 처리한 후, 구한 값을 [제출 형식]에 맞춰 답안을 입력하시오.

**[수행 순서]**
1. 나이(`age`)를 10살 단위로 묶어 연령대(10대, 20대, 30대, 40대, 50대, 60대)로 구분하시오. (예: 25세 → 20대, 41세 → 40대)
2. 연령대별 평균 소비 금액(`spending`)을 계산하고, 평균 소비 금액이 가장 높은 연령대를 찾으시오.
3. 해당 연령대에 속하는 고객의 수(인원수)를 구하시오.

**제공 데이터 (customer_age.csv)**
| 컬럼명 | 설명 |
|---|---|
| customer_id | 고객 ID |
| age | 나이 |
| spending | 소비 금액(원) |

**[제출 형식]**
- 정수로 작성

### 문제 풀이 방향

지금까지는 이미 데이터에 존재하는 범주(부서, 성과등급, 대륙, 카테고리 등)를 기준으로 그룹을 나눴지만, 이 문제는 나이라는 연속된 숫자 데이터를 먼저 10대, 20대 같은 범주로 직접 나눠야 한다는 점이 새롭습니다. 이렇게 숫자를 구간으로 나누는 작업을 구간화(binning)라고 부릅니다. 구간화를 마친 뒤에는 그 범주를 기준으로 평균을 구하고 순위를 찾는, 앞서 계속 해온 방식과 동일한 흐름으로 이어집니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [17]:
import pandas as pd

df = pd.read_csv('customer_age.csv')
print(df.head())

  customer_id  age  spending
0       C0001   19     51732
1       C0002   53     65593
2       C0003   58     55092
3       C0004   27     66458
4       C0005   58     71059


**셀 2) 나이를 10살 단위 연령대로 구간화하기**

In [18]:
bins = [10, 20, 30, 40, 50, 60, 70]
labels = ['10대', '20대', '30대', '40대', '50대', '60대']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

print(df[['age', 'age_group']].head())

   age age_group
0   19       10대
1   53       50대
2   58       50대
3   27       20대
4   58       50대


`pd.cut()`은 연속된 숫자 값을 지정한 구간으로 나눠서 범주형 데이터로 바꿔주는 함수입니다. `bins=[10, 20, 30, 40, 50, 60, 70]`은 경계값을 나열한 것으로, 10에서 20 사이, 20에서 30 사이처럼 총 6개의 구간을 정의합니다. `labels`는 각 구간에 붙일 이름으로, 첫 번째 구간(10 ~ 20)에는 10대, 두 번째 구간(20 ~ 30)에는 20대라는 이름이 순서대로 대응됩니다.

`right=False`라는 옵션을 짚어볼 필요가 있습니다. 기본적으로 pd.cut은 구간의 오른쪽 끝(경계값)을 포함하도록 되어 있어서, 만약 이 옵션 없이 그대로 사용하면 정확히 20살인 사람이 10대 구간(10 초과 20 이하)에 포함되어버립니다. 하지만 우리가 원하는 것은 20살이 20대로 분류되는 것이므로, `right=False`를 지정해서 구간의 왼쪽 끝을 포함하고 오른쪽 끝은 포함하지 않도록(10 이상 20 미만) 바꿔주었습니다. 즉 정확히 20살인 사람은 10대가 아니라 20대 구간에 들어가게 됩니다. 이렇게 경계에 걸친 값이 어느 구간으로 분류되는지는 함수의 옵션에 따라 달라지므로, 문제에서 제시한 예시(25세는 20대, 41세는 40대)와 실제 결과가 일치하는지 확인해보는 습관이 중요합니다.

> 💡 pd.cut은 구간의 개수와 경계를 직접 지정할 때 사용하고, 만약 데이터를 정확히 몇 개의 그룹으로 균등하게(예: 4분위로) 나누고 싶다면 `pd.qcut()`이라는 함수를 사용합니다. 두 함수는 이름은 비슷하지만 구간을 나누는 기준이 다릅니다.

**셀 3) 연령대별 평균 소비 금액 구하고 가장 높은 연령대 찾기 (수행 순서 2)**

In [19]:
group_mean = df.groupby('age_group', observed=True)['spending'].mean().sort_values(ascending=False)
print(group_mean)

top_group = group_mean.idxmax()
print("평균 소비금액이 가장 높은 연령대:", top_group)

age_group
40대    96458.380952
50대    69496.971429
30대    67800.602410
20대    55501.937500
60대    51803.206897
10대    31641.541667
Name: spending, dtype: float64
평균 소비금액이 가장 높은 연령대: 40대


구간화가 끝난 뒤에는 age_group이라는 새로운 범주형(Categorical) 컬럼이 생겼으므로, 이 컬럼을 기준으로 `groupby('age_group')['spending'].mean()`을 적용하면 연령대별 평균 소비 금액이 계산됩니다. 여기까지는 앞서 대륙별 평균, 부서별 평균을 구했던 것과 완전히 같은 방식입니다. `idxmax()`로 평균이 가장 큰 연령대의 이름을 가져옵니다.

여기서 `observed=True`라는 옵션을 하나 짚어보겠습니다. pd.cut으로 만든 컬럼은 일반 문자열이 아니라 범주형(Categorical)이라는 특별한 타입인데, 이 타입으로 groupby를 하면 pandas 버전에 따라 다음과 같은 경고가 뜰 수 있습니다.

```
FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas.
```

이 경고는 groupby의 기준이 범주형일 때, 실제 데이터에 등장하지 않는 범주까지 그룹으로 포함시킬지(`observed=False`) 아니면 실제로 데이터에 등장한 범주만 그룹으로 만들지(`observed=True`)를 아직 명시하지 않았다는 뜻입니다. 지금 데이터는 10대부터 60대까지 6개 범주가 모두 실제로 등장하므로 결과 자체는 어느 쪽을 선택해도 같지만, 옵션을 명시하지 않으면 경고 메시지가 출력됩니다. `observed=True`를 직접 지정해두면 경고 없이 실행되고, 앞으로 바뀔 pandas의 기본 동작과도 미리 맞춰두는 셈이 됩니다.

**셀 4) 해당 연령대의 인원수 구하기 (정답)**

In [20]:
answer = (df['age_group'] == top_group).sum()
print("정답:", answer)

정답: 105


`df['age_group'] == top_group`은 age_group이 '40대'인지 아닌지를 행마다 True 또는 False로 판별한 결과입니다. 파이썬에서 True는 숫자 1, False는 숫자 0으로 취급되므로, 이 True/False 값들을 `.sum()`으로 모두 더하면 조건을 만족하는(즉 True인) 행의 개수, 다시 말해 40대에 속하는 고객의 수가 그대로 계산됩니다. 이는 조건을 만족하는 행이 몇 개인지 세고 싶을 때 자주 사용하는 방법입니다.

### 전체 코드

In [16]:
import pandas as pd

df = pd.read_csv('customer_age.csv')

bins = [10, 20, 30, 40, 50, 60, 70]
labels = ['10대', '20대', '30대', '40대', '50대', '60대']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

group_mean = df.groupby('age_group', observed=True)['spending'].mean()
top_group = group_mean.idxmax()

answer = (df['age_group'] == top_group).sum()
print("정답:", answer)

정답: 105


### 정리

이 문제에서 새롭게 등장한 것은 pd.cut을 이용한 구간화와, right=False라는 경계값 처리 옵션이었습니다. 나이처럼 연속된 숫자를 그대로 그룹 기준으로 쓸 수는 없으므로, 먼저 의미 있는 범주로 변환하는 과정이 필요하다는 것을 확인했습니다. 한 가지 더 짚을 부분은, `(조건).sum()`으로 조건을 만족하는 행의 개수를 세는 방법입니다. 이 방법은 구간화 문제뿐 아니라 특정 조건을 만족하는 데이터가 몇 건인지 세어야 하는 모든 상황에서 반복해서 쓰이는 패턴이므로 꼭 기억해두는 것이 좋습니다.

---

## 이번 회차에서 사용한 핵심 함수 정리
| 상황 | 함수 | 핵심 이유 |
|---|---|---|
| 그룹별로 조건에 맞는 행(최댓값을 가진 행)을 각각 찾기 | `groupby(기준)[대상].idxmax()` + `df.loc[인덱스]` | groupby로 나눈 뒤에는 뒤따르는 함수가 각 그룹 안에서 독립적으로 동작 |
| 텍스트를 숫자로 변환하기 (단어 개수 등) | `.apply(lambda x: ...)` | 내장 함수로 처리되지 않는 개별적인 로직을 컬럼 전체에 적용 |
| 문자열을 특정 기준으로 나누기 | `문자열.split(구분자)` | 공백, 쉼표 등 원하는 기준으로 문자열을 리스트로 분리 |
| Series에서 이름(인덱스)으로 값 조회 | `Series['이름']` | groupby 결과처럼 이름이 인덱스인 Series에서 특정 값만 조회 |
| 연속된 숫자를 구간(범주)으로 나누기 | `pd.cut(컬럼, bins=경계리스트, labels=이름리스트, right=False)` | 경계값이 어느 구간에 속하는지는 right 옵션으로 결정 |
| 범주형(Categorical) 컬럼으로 groupby하기 | `groupby(컬럼, observed=True)` | 실제 데이터에 등장한 범주만 그룹으로 사용, 경고 방지 |
| 조건을 만족하는 행의 개수 세기 | `(조건).sum()` | True/False가 1/0으로 취급되어 합계가 곧 개수가 됨 |

---

# 작업형 제2유형

## 문제

제공된 학습용 데이터(train.csv)는 서울시 아파트 단지의 월별 관리비 데이터이다. 학습용 데이터를 활용하여 단지별 월 관리비(management_fee)를 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 관리비를 예측하시오. 예측 결과는 아래의 [유의사항]과 [제출 형식]을 준수하여, CSV 파일(result.csv)을 생성하는 코드를 제출하시오.

**[유의사항]**
- ① 예측 결과는 MAE(Mean Absolute Error) 평가지표에 따라 평가함
- ② 타겟 변수(management_fee)는 일부 값이 0으로 기재되어 있으며, 이는 결측치를 대체한 값이다

**[제출 형식]**
- ① CSV 파일명: result.csv
- ② 예측 관리비 컬럼명: pred
- ③ 제출 컬럼 개수: pred 컬럼 1개
- ④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 1,080개

**[제공 데이터]**
- train.csv: 학습용 데이터, 약 2,900개
- test.csv: 평가용 데이터, 약 1,080개

| 컬럼명 | 설명 |
|---|---|
| id | 단지 ID |
| total_units | 총 세대수 |
| building_age | 건축 연식(년) |
| area_type | 면적 유형 (소형/중형/대형) |
| heating_type | 난방 방식 (개별난방/중앙난방/지역난방) |
| elevator_count | 승강기 수 |
| parking_ratio | 세대당 주차대수 비율 |
| region | 권역 (강남권/강북권/강서권/강동권) |
| month | 조사 월 (1~12) |
| management_fee | (train에만 존재) 월 관리비(원) |

### 문제 풀이 방향

이 문제에서 가장 먼저 신경 써야 할 것은 유의사항 ②번입니다. management_fee 중 일부가 0으로 기재되어 있는데, 이는 실제로 관리비가 0원이라는 뜻이 아니라 결측치를 0으로 표시해둔 것이라고 명시되어 있습니다. 이걸 그냥 지나치고 0을 진짜 관리비 값으로 취급해서 학습에 사용하면, 모델은 "이런 조건이면 관리비가 0원"이라는 잘못된 패턴을 배우게 됩니다. 그래서 본격적인 모델링에 앞서 이 0값들을 어떻게 다룰지 먼저 판단해야 합니다.

또한 area_type, heating_type, region처럼 순서나 크기 관계가 없는(예: 대형이 소형보다 숫자가 큰 것은 아님) 범주형 변수를 다루는 방법으로, 이번에는 LabelEncoder 대신 get_dummies(원-핫 인코딩)를 사용해보겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [1]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()

(2900, 10) (1080, 9)


,id,total_units,building_age,area_type,heating_type,elevator_count,parking_ratio,region,month,management_fee
0,1508,1232,28,소형,개별난방,9,1.10,강남권,7,135905.0
1,1451,263,3,대형,개별난방,11,1.64,강동권,11,204782.0
2,1623,614,29,중형,개별난방,7,1.03,강북권,10,167558.0
3,160,1005,0,중형,지역난방,2,1.17,강서권,4,158636.0
4,2936,422,28,중형,개별난방,2,1.61,강북권,1,181583.0


train은 10개 컬럼(management_fee 포함), test는 9개 컬럼(management_fee 없음)입니다. management_fee가 우리가 예측해서 채워야 할 대상이기 때문에 test에는 제공되지 않습니다. `train.shape`과 `test.shape`을 먼저 확인해두면, 이후 제출 파일의 행 개수가 test의 행 개수(1,080개)와 정확히 일치하는지 확인할 때 기준이 됩니다.

**셀 2) 타겟 변수 확인 (0값 존재 여부 포함)**

In [2]:
print(train.isna().sum().sum())
print(test.isna().sum().sum())

0
0


train와 test 모두 일반적인 결측치(NaN)는 없습니다.

In [3]:
print(train['management_fee'].describe())
print("0으로 기재된 행 수:", (train['management_fee'] == 0).sum())

count      2900.000000
mean     157996.566897
std       58068.095162
min           0.000000
25%      124859.750000
50%      157057.000000
75%      189497.000000
max      295683.000000
Name: management_fee, dtype: float64
0으로 기재된 행 수: 145


하지만 `describe()`의 최솟값이 0이고 실제로 145개 행이 정확히 0이라는 것을 확인할 수 있습니다. 관리비 평균이 15만 원대인 것을 생각하면, 0원짜리 관리비가 145건이나 있다는 것은 상식적으로 이상합니다. 유의사항에서 이미 이 값들이 결측치 대체값이라고 알려주었으므로, 이 145건은 진짜 데이터가 아니라 사실상 결측치라고 판단하는 것이 타당합니다.

**셀 3) 결측치로 판단되는 0값 행을 학습에서 제외**

In [4]:
train_clean = train[train['management_fee'] > 0].reset_index(drop=True)
print("제외 전:", train.shape, "제외 후:", train_clean.shape)

제외 전: (2900, 10) 제외 후: (2755, 10)


`train['management_fee'] > 0`으로 관리비가 0보다 큰 행만 남기고, 결측치가 대체된 것으로 보이는 145개 행은 학습 데이터에서 제외했습니다. 이 판단에는 다른 선택지도 있을 수 있습니다. 예를 들어 0을 다른 값으로 대체하는 방법도 고려할 수 있지만, 이 값들이 애초에 신뢰할 수 없는 값이라는 점을 생각하면 학습에서 제외하는 편이 모델을 왜곡시키지 않는 안전한 접근입니다. 참고로 이 처리는 학습(train) 데이터에만 적용합니다. test 데이터는 애초에 management_fee 컬럼 자체가 제공되지 않으므로 이런 판단이 필요하지 않습니다.

> 💡 문제의 유의사항에 타겟 변수나 특정 컬럼에 대한 데이터 품질 관련 안내(0으로 대체됨, 특정 값은 오류임 등)가 있다면, 그 값을 무비판적으로 그대로 사용하지 말고 어떻게 처리할지 먼저 판단하는 것이 중요합니다. 이런 안내는 대체로 문제를 푸는 사람이 반드시 짚고 넘어가야 할 지점을 알려주는 힌트입니다.

**셀 4) 범주형 변수 원-핫 인코딩 (get_dummies)**

In [5]:
cat_cols = ['area_type', 'heating_type', 'region']
train_dum = pd.get_dummies(train_clean, columns=cat_cols)
test_dum = pd.get_dummies(test, columns=cat_cols)

print(train_dum.columns.tolist())

['id', 'total_units', 'building_age', 'elevator_count', 'parking_ratio', 'month', 'management_fee', 'area_type_대형', 'area_type_소형', 'area_type_중형', 'heating_type_개별난방', 'heating_type_중앙난방', 'heating_type_지역난방', 'region_강남권', 'region_강동권', 'region_강북권', 'region_강서권']


`get_dummies()`는 범주형 컬럼 하나를, 그 컬럼에 등장하는 범주 개수만큼의 새로운 컬럼으로 펼쳐줍니다. 예를 들어 area_type(소형/중형/대형)은 `area_type_소형`, `area_type_중형`, `area_type_대형`이라는 3개의 True/False 컬럼으로 바뀝니다. 어떤 행의 area_type이 원래 '소형'이었다면, `area_type_소형`만 True이고 나머지는 False가 됩니다.

LabelEncoder와의 차이를 짚어볼 필요가 있습니다. LabelEncoder는 '소형'=0, '중형'=1, '대형'=2처럼 순서가 있는 하나의 숫자로 바꾸는데, 이렇게 하면 모델이 "대형(2)이 소형(0)보다 크다"는 식으로 크기 관계가 있다고 잘못 해석할 위험이 있습니다. 실제로 순서에 의미가 있는 값(예: 저위험<중위험<고위험)이라면 이런 인코딩도 자연스럽지만, region처럼 강남권과 강북권 사이에 크고 작음이라는 개념이 없는 순수한 명목형 변수라면 get_dummies로 각 범주를 독립적인 컬럼으로 분리하는 것이 더 적합합니다.

**셀 5) train과 test의 컬럼 구성 맞추기**

In [6]:
test_dum = test_dum.reindex(columns=train_dum.drop(columns=['management_fee']).columns, fill_value=0)

print(test_dum.shape)
test_dum.head()

(1080, 16)


,id,total_units,building_age,elevator_count,parking_ratio,month,area_type_대형,area_type_소형,area_type_중형,heating_type_개별난방,heating_type_중앙난방,heating_type_지역난방,region_강남권,region_강동권,region_강북권,region_강서권
0,3166,1282,26,7,0.93,10,False,False,True,False,True,False,False,False,False,True
1,3169,110,17,1,1.61,5,False,True,False,False,True,False,True,False,False,False
2,3043,684,39,1,0.66,2,False,False,True,True,False,False,True,False,False,False
3,1273,1019,16,5,0.73,8,False,False,True,True,False,False,False,False,True,False
4,3126,218,16,4,1.36,11,False,True,False,False,True,False,False,False,False,True


get_dummies를 사용할 때 반드시 주의해야 할 부분이 있습니다. train과 test를 각각 따로 get_dummies에 넣으면, 혹시라도 한쪽에는 등장하지만 다른 쪽에는 등장하지 않는 범주가 있을 경우 두 데이터의 컬럼 구성(개수와 순서)이 서로 달라질 수 있습니다. 예를 들어 test 데이터에 우연히 특정 지역 데이터가 하나도 없다면, test_dum에는 그 지역에 해당하는 컬럼 자체가 생기지 않습니다. 이 상태로 모델에 넣으면 컬럼 개수가 맞지 않아 에러가 발생합니다.

`reindex(columns=..., fill_value=0)`는 test_dum의 컬럼을 train_dum과 동일한 목록, 동일한 순서로 강제로 맞춰줍니다. train에는 있지만 test에는 없던 컬럼은 값이 0(해당 범주가 아님)으로 채워져 새로 생기고, 순서도 train과 정확히 같아집니다. 이 문제에서는 우연히 모든 범주가 양쪽에 다 등장했지만, 이런 안전장치는 습관적으로 넣어두는 것이 좋습니다.

> 💡 get_dummies를 쓸 때는 train과 test를 합친 뒤 한 번에 get_dummies를 적용하고 나중에 다시 나누는 방법도 있습니다. 다만 이 경우 train에 없던 값이 test에만 있을 때도 문제가 될 수 있으므로, `reindex`로 train 기준 컬럼에 test를 맞추는 방식이 더 안전하고 명확합니다.

**셀 6) 학습용/검증용 데이터 분리**

In [7]:
from sklearn.model_selection import train_test_split

X = train_dum.drop(columns=['id', 'management_fee'])
y = train_dum['management_fee']
X_test = test_dum.drop(columns=['id'])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_tr.shape, X_val.shape)

(2204, 15) (551, 15)


`id`는 단지를 구분하기 위한 값일 뿐 관리비와 직접적인 관련이 없으므로 학습에 사용할 컬럼(X)에서 제외했습니다. `train_test_split`으로 train_dum의 일부(20%)를 검증용으로 떼어둔 이유는, 진짜 test.csv에는 정답(management_fee)이 없어서 우리가 만든 모델이 얼마나 정확한지 미리 확인할 방법이 없기 때문입니다. 학습 데이터 중 일부를 검증용으로 남겨두면, 그 부분에 대해서는 정답을 알고 있으므로 모델의 예측과 실제값을 비교해서 점수를 미리 가늠해볼 수 있습니다. management_fee는 클래스가 아니라 연속적인 숫자이므로, 분류 문제에서 사용했던 stratify 옵션은 여기서는 사용하지 않았습니다.

**셀 7) 모델 학습 및 검증 성능 확인**

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)
print("검증 MAE:", mean_absolute_error(y_val, val_pred))

검증 MAE: 7519.48221415608


`mean_absolute_error`는 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균을 낸 값입니다. 예를 들어 실제 관리비가 150,000원인데 140,000원으로 예측했다면 오차는 10,000원이고, 반대로 160,000원으로 예측했어도 오차는 똑같이 10,000원입니다. 이렇게 예측이 실제보다 높든 낮든 방향과 상관없이 오차의 크기만 보고, 그 크기들을 단순 평균 낸 것이 MAE입니다.

MAE와 자주 비교되는 지표로 RMSE가 있는데, RMSE는 오차를 절댓값이 아니라 제곱해서 평균 낸 뒤 다시 제곱근을 씌우는 방식입니다. 오차를 제곱하면 큰 오차일수록 그 값이 훨씬 더 크게 부풀려지기 때문에, RMSE는 몇 개의 아주 큰 오차(이상치)에 특히 민감하게 반응합니다. 반면 MAE는 제곱 없이 절댓값만 사용하므로 모든 오차를 원래 크기 그대로, 동일한 비중으로 반영합니다. 이 문제에서는 MAE가 요구되었으므로 `mean_absolute_error`를 사용했습니다. 검증 MAE가 약 7,519라는 것은, 평균적으로 예측이 실제 관리비와 약 7,500원 정도 차이 난다는 뜻이며, 관리비 평균이 15만 원대인 것을 고려하면 준수한 수준입니다.

**셀 8) 전체 학습 데이터로 재학습 후 test 예측**

In [9]:
model.fit(X, y)
pred = model.predict(X_test)

print(pred[:10])

[176292.88       123816.72666667 199613.88666667 173718.93666667
 124805.63333333 259130.49333333 186196.47666667 265558.23
 152505.9        243558.58333333]


이 예측값에는 음수를 0으로 보정하는 별도의 후처리를 적용하지 않았습니다. 관리비처럼 현실적으로 음수가 나올 수 없는 값을 예측할 때는, 모델이 훈련 데이터에 없던 음수를 예측하지 않는지 확인하는 습관이 필요합니다. RandomForest 계열의 모델은 훈련 데이터에서 관찰한 값들의 범위 안에서 예측하는 특성이 있어 음수를 예측할 가능성이 거의 없지만, 만약 선형회귀 계열의 모델을 사용한다면 특성 조합에 따라 이론적으로 음수가 나올 수도 있습니다. 이런 경우에는 `np.clip(pred, 0, None)`처럼 예측값의 하한을 0으로 강제해주는 보정이 필요합니다. 이번 모델은 실제로 예측값을 확인했을 때 전부 상식적인 범위 안에 있었으므로 별도의 보정 없이 그대로 사용했습니다.

**셀 9) 제출 파일 생성**

In [10]:
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)

print(result.shape)
print(result.head())

(1080, 1)
            pred
0  176292.880000
1  123816.726667
2  199613.886667
3  173718.936667
4  124805.633333


제출 형식에서 요구한 대로 `pred`라는 컬럼 하나만 있는 데이터프레임을 만들고, `to_csv()`로 저장했습니다. `index=False`를 반드시 넣어야 합니다. 이 옵션이 없으면 행 번호(0, 1, 2, ...)가 별도의 컬럼으로 파일에 함께 저장되어, 요구된 pred 컬럼 1개라는 조건을 어기게 됩니다.

저장 후에는 `result.shape`으로 행 개수가 test 데이터 개수(1,080개)와 정확히 일치하는지 확인했습니다. 예측값이 176292.88처럼 소수 형태로 나온 것은 회귀 모델의 예측이 연속적인 값이기 때문이며, 관리비를 정수(원 단위)로 반올림하라는 별도 지시가 없었으므로 소수 형태 그대로 제출해도 무방합니다.

### 전체 코드

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 결측치가 0으로 대체된 것으로 판단되는 행을 학습에서 제외
train_clean = train[train['management_fee'] > 0].reset_index(drop=True)

# 범주형 변수 원-핫 인코딩
cat_cols = ['area_type', 'heating_type', 'region']
train_dum = pd.get_dummies(train_clean, columns=cat_cols)
test_dum = pd.get_dummies(test, columns=cat_cols)

# test의 컬럼을 train 기준으로 맞추기
test_dum = test_dum.reindex(columns=train_dum.drop(columns=['management_fee']).columns, fill_value=0)

X = train_dum.drop(columns=['id', 'management_fee'])
y = train_dum['management_fee']
X_test = test_dum.drop(columns=['id'])

# 검증용 분리 후 성능 확인
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print("검증 MAE:", mean_absolute_error(y_val, val_pred))

# 전체 데이터로 재학습 후 test 예측
model.fit(X, y)
pred = model.predict(X_test)

# 제출 파일 생성
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

검증 MAE: 7519.48221415608
(1080, 1)


### 정리

이 문제의 핵심은 두 가지였습니다. 첫째는 유의사항에 적힌 "0은 결측치를 대체한 값"이라는 문장을 무시하지 않고, 그 값을 학습에서 제외하는 판단을 내린 것입니다. 이런 데이터 품질에 대한 안내를 놓치면 모델이 잘못된 패턴을 학습하게 됩니다. 둘째는 get_dummies로 순서 없는 범주형 변수를 인코딩하면서, train과 test의 컬럼 구성을 reindex로 맞춰야 한다는 것이었습니다. LabelEncoder는 한 컬럼을 그대로 숫자로 바꾸기 때문에 이런 컬럼 개수 문제가 생기지 않지만, get_dummies는 컬럼 개수 자체가 늘어나는 방식이라 이 차이를 반드시 관리해야 합니다.

이번 모델의 검증 MAE는 약 7,519원 수준이었습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 타겟 변수에 결측치가 특정 값(0 등)으로 대체되어 있을 때 | 해당 행을 학습에서 제외 (`df[df[col] > 0]`) | 잘못된 값을 그대로 학습하면 모델이 왜곡됨 |
| 순서 없는(명목형) 범주형 변수 인코딩 | `pd.get_dummies(df, columns=[...])` | 범주 간 크기 관계가 없다고 모델에 알려주는 인코딩 방식 |
| get_dummies 사용 후 train/test 컬럼 정렬 | `test.reindex(columns=train.columns, fill_value=0)` | 양쪽에 등장하는 범주가 다르면 컬럼 개수가 어긋날 수 있음 |
| 회귀 평가지표(RMSE와의 차이) | `mean_absolute_error(y, pred)` | 오차를 제곱하지 않고 절댓값으로만 평가, 큰 오차에 상대적으로 덜 민감 |

---

# 작업형 제3유형

## 문제 1. (10점)

제공된 데이터(diet_program.csv)는 다이어트 프로그램에 참여한 50명의 참가자를 대상으로, 프로그램 시작 전과 8주 후의 체중을 측정한 자료이다. 프로그램 참여 전후로 체중에 통계적으로 유의한 차이가 있는지 대응표본 t검정(paired t-test)을 수행하고, 아래 소문항에 답하시오.

**제공 데이터 (diet_program.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 참가자 번호 |
| weight_before | 프로그램 시작 전 체중(kg) |
| weight_after | 프로그램 참여 8주 후 체중(kg) |

**문제 1-1** 프로그램 참여 후 체중과 참여 전 체중의 차이(after - before)의 평균을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 1-2** 대응표본 t검정을 수행하고, t통계량을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 1-3** 위 검정의 p-value를 구하고, 유의수준 0.05에서 귀무가설을 기각하는지 채택하는지 판단하시오.

### 문제 풀이 방향

대응표본 t검정은 같은 대상(사람, 사물 등)을 두 번 측정해서 그 차이를 비교할 때 사용하는 검정입니다. 이 데이터는 같은 50명의 참가자를 프로그램 전과 후, 두 시점에 각각 측정했으므로 전형적인 대응표본 상황입니다. 귀무가설(H0)은 "프로그램 전후 체중에 차이가 없다(차이의 평균이 0이다)"이고, 대립가설(H1)은 "차이가 있다"입니다. 만약 이 데이터가 서로 다른 두 그룹의 사람들(예: 프로그램 참여자 50명 vs 비참여자 50명)을 비교하는 것이었다면 독립표본 t검정을 사용해야 하는데, 지금은 동일 인물을 반복 측정한 것이므로 대응표본 t검정이 적절합니다. 먼저 차이값 자체를 눈으로 확인한 뒤, `scipy.stats`의 `ttest_rel()` 함수로 검정을 수행하겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [1]:
import pandas as pd

df = pd.read_csv('diet_program.csv')
print(df.shape)
df.head()

(50, 3)


,id,weight_before,weight_after
0,1,86.8,82.6
1,2,70.6,68.9
2,3,83.6,80.3
3,4,73.9,70.9
4,5,89.8,90.7


50명의 참가자 데이터이며, 한 사람당 weight_before(전)와 weight_after(후) 두 개의 측정값이 같은 행에 나란히 기록되어 있습니다. 이렇게 같은 대상의 두 측정값이 한 행에 짝지어져 있는 형태가 대응표본 데이터의 전형적인 모습입니다. (참가자 5번처럼 오히려 체중이 늘어난 경우도 있는데, 이는 개인차이며 대응표본 t검정은 이런 개별 편차를 감안하면서도 전체적인 평균 변화가 우연이라고 보기 어려운 수준인지를 판단합니다.)

**셀 2) 전후 체중 차이 계산 (문제 1-1 정답)**

In [2]:
diff = df['weight_after'] - df['weight_before']
print("평균 차이:", round(diff.mean(), 3))
print(diff.describe())

평균 차이: -3.586
count    50.000000
mean     -3.586000
std       2.429832
min      -8.700000
25%      -5.075000
50%      -3.250000
75%      -1.850000
max       1.000000
dtype: float64


`weight_after - weight_before`처럼 두 컬럼을 그대로 빼면, 같은 행끼리(즉 같은 참가자끼리) 짝지어 계산되어 50명 각각의 체중 변화량이 구해집니다. 평균이 -3.586이라는 것은, 참가자들이 평균적으로 약 3.586kg 감량했다는 뜻입니다. `describe()`로 전체 분포를 함께 살펴보면, 최소값이 -8.7(가장 많이 감량한 사람)이고 최댓값이 1.0(체중이 늘어난 사람도 있음)이라는 것도 확인할 수 있습니다. 평균만으로는 "감량 효과가 있어 보인다" 정도만 알 수 있을 뿐, 이 차이가 우연히 생긴 것인지 통계적으로 의미 있는 것인지는 다음 셀의 검정을 통해 확인해야 합니다.

**셀 3) 대응표본 t검정 수행**

In [3]:
from scipy.stats import ttest_rel

t_stat, p = ttest_rel(df['weight_after'], df['weight_before'])
print("t통계량:", t_stat)
print("p-value:", p)

t통계량: -10.435636917264883
p-value: 4.79534825979184e-14


`ttest_rel(a, b)`는 대응표본 t검정을 수행하는 함수로, 내부적으로는 `a - b`의 차이값들을 가지고 "이 차이들의 평균이 0과 다르다고 할 수 있는가"를 검정합니다. 인자의 순서(먼저 after, 다음 before)에 따라 t통계량의 부호가 달라지므로, "무엇에서 무엇을 뺀 차이인지"를 문제의 지시와 일치시켜 순서를 정하는 것이 중요합니다. 문제에서 "참여 후 체중과 참여 전 체중의 차이(after - before)"라고 명시했으므로, `ttest_rel(weight_after, weight_before)`처럼 after를 먼저 넣었습니다.

t통계량이 음수(-10.44)로 나온 것은 셀 2에서 확인한 평균 차이가 음수(체중이 감소)였던 것과 같은 방향입니다. t통계량의 절댓값이 클수록(0에서 멀수록) "차이가 0이라는 가정과 실제 데이터가 크게 어긋난다"는 뜻이므로, 통계적으로 유의한 차이일 가능성이 높아집니다.

**셀 4) t통계량 확인 (문제 1-2 정답)**

In [4]:
print("1-2 정답:", round(t_stat, 3))

1-2 정답: -10.436


> 💡 t통계량은 "차이의 평균을 그 차이의 표준오차(standard error)로 나눈 값"입니다. 즉 단순히 차이의 크기(-3.586kg)만 보는 것이 아니라, 그 차이가 참가자들 사이에서 얼마나 일관되게 나타났는지(개인차가 작을수록 표준오차도 작아짐)까지 함께 고려한 값입니다. 평균 차이가 크더라도 사람마다 들쭉날쭉한 정도(표준편차)가 크다면 t통계량은 작아질 수 있고, 반대로 평균 차이가 작더라도 모든 사람이 일관되게 비슷한 변화를 보였다면 t통계량은 커질 수 있습니다.

**셀 5) p-value 확인 및 기각/채택 판단 (문제 1-3 정답)**

In [5]:
alpha = 0.05

if p < alpha:
    conclusion = "귀무가설 기각(프로그램 전후 체중에 유의한 차이가 있다)"
else:
    conclusion = "귀무가설 채택(프로그램 전후 체중에 유의한 차이가 없다)"

print("p-value:", p)
print("1-3 정답:", conclusion)

p-value: 4.79534825979184e-14
1-3 정답: 귀무가설 기각(프로그램 전후 체중에 유의한 차이가 있다)


p-value가 약 0.0000000000000479로 유의수준 0.05보다 극히 작으므로, 귀무가설(전후 체중에 차이가 없다)을 기각합니다. 즉 이 다이어트 프로그램은 참여 전후로 체중에 통계적으로 유의한 변화(감소)를 가져왔다고 결론 내릴 수 있습니다. p-value가 이렇게 극단적으로 작게 나온 이유는, 셀 2의 `describe()` 결과에서 확인했듯 50명 대부분이 비슷한 방향(감소)으로 변화했고 개인차(표준편차 2.43)에 비해 평균 변화(-3.586)가 상당히 뚜렷했기 때문입니다.

### 전체 코드

In [6]:
import pandas as pd
from scipy.stats import ttest_rel

df = pd.read_csv('diet_program.csv')

# 1-1: 전후 차이의 평균
diff = df['weight_after'] - df['weight_before']
print("1-1 정답:", round(diff.mean(), 3))

# 대응표본 t검정
t_stat, p = ttest_rel(df['weight_after'], df['weight_before'])

# 1-2: t통계량
print("1-2 정답:", round(t_stat, 3))

# 1-3: p-value 및 판단
alpha = 0.05
conclusion = "귀무가설 기각" if p < alpha else "귀무가설 채택"
print("1-3 정답: p-value =", p, "->", conclusion)

1-1 정답: -3.586
1-2 정답: -10.436
1-3 정답: p-value = 4.79534825979184e-14 -> 귀무가설 기각


### 정리

대응표본 t검정은 같은 대상을 두 번 측정한 데이터에서, 그 차이의 평균이 통계적으로 0과 다르다고 할 수 있는지를 검정하는 방법입니다. 핵심은 두 값을 빼서 하나의 차이값으로 만든 뒤, 그 차이값들의 평균과 개인차(산포)를 함께 고려해 t통계량을 계산한다는 것입니다. 지금까지 다룬 카이제곱 검정이나 ANOVA와 마찬가지로, 결국 최종 판단은 p-value를 유의수준과 비교하는 동일한 절차로 이루어진다는 점도 다시 확인할 수 있었습니다.

---

## 문제 2. (10점)

제공된 데이터(promotion_conversion.csv)는 온라인 리테일 프로모션에 노출된 고객의 행동 데이터이다. 나이, 웹사이트 체류 시간, 과거 구매 횟수, 이메일 열람 횟수, 할인율 등에 따른 프로모션 구매 전환 여부(converted)를 예측하기 위해 로지스틱 회귀분석을 수행하고, 아래 소문항에 답하시오.

**제공 데이터 (promotion_conversion.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 고객 번호 |
| age | 나이 |
| browsing_time | 웹사이트 체류 시간(분) |
| past_purchases | 과거 구매 횟수 |
| email_open_count | 프로모션 이메일 열람 횟수 |
| discount_offered | 제공된 할인율(%) |
| converted | (종속변수) 구매 전환 여부 — 0: 미전환, 1: 전환 |

**문제 2-1** id를 제외한 모든 변수를 독립변수로 사용하여 로지스틱 회귀모형을 적합하시오. 유의확률(p-value)이 0.05보다 작은 변수만 사용하여 모형을 다시 적합하고, 이 모형의 회귀계수(절편 포함)의 합을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-2** 위에서 재적합한 모형을 기준으로, discount_offered가 5%p 증가할 때의 오즈비(Odds Ratio)를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-3** 위 모형으로 전체 데이터의 구매 전환 확률을 예측했을 때, 예측 확률이 0.3을 초과하는 고객의 수를 구하시오. (정수로 작성)

### 문제 풀이 방향

이 문제는 지금까지 다룬 로지스틱 회귀 문제들의 요소를 한 번에 종합합니다. 유의한 변수만 골라 재적합하는 과정, 계수의 합을 구하는 것, 오즈비를 해석하는 것, 그리고 예측 확률을 기준으로 인원수를 세는 것까지 모두 포함되어 있습니다. 다만 문제 2-2에서는 변수가 1단위가 아니라 5단위 증가할 때의 오즈비를 구해야 한다는 점이 새롭습니다. 이 부분을 어떻게 계산하는지가 이번 문제의 핵심 포인트입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [7]:
import pandas as pd

df = pd.read_csv('promotion_conversion.csv')
print(df.shape)
df.head()

(900, 7)


,id,age,browsing_time,past_purchases,email_open_count,discount_offered,converted
0,1,62,19.3,4,5,25,1
1,2,50,0.5,2,5,25,0
2,3,30,4.1,1,4,20,0
3,4,33,0.9,1,6,20,0
4,5,29,5.5,3,4,15,1


900명의 고객 데이터입니다. 이번 문제는 id를 제외한 모든 컬럼이 이미 숫자형이므로 별도의 범주형 인코딩이 필요하지 않습니다.

**셀 2) 전체 변수로 로지스틱 회귀모형 적합**

In [9]:
import statsmodels.api as sm

X = df.drop(columns=['id', 'converted'])
y = df['converted']
X = sm.add_constant(X)

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.597497
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:                  900
Model:                          Logit   Df Residuals:                      894
Method:                           MLE   Df Model:                            5
Date:                Thu, 09 Jul 2026   Pseudo R-squ.:                  0.1010
Time:                        15:56:37   Log-Likelihood:                -537.75
converged:                       True   LL-Null:                       -598.14
Covariance Type:            nonrobust   LLR p-value:                 2.129e-24
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -2.8605      0.351     -8.139      0.000      -3.549      -2.172
age        

계수표를 보면 age를 제외한 네 변수(browsing_time, past_purchases, email_open_count, discount_offered)는 모두 p-value가 0.05보다 작아 유의하지만, age는 p-value가 0.534로 전혀 유의하지 않습니다. 즉 이 데이터 안에서는 나이가 구매 전환 여부를 설명하는 데 통계적으로 의미 있는 역할을 하지 못하는 것으로 나타났습니다.

**셀 3) 유의한 변수 선별**

In [10]:
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print("유의한 변수(절편 포함):", sig_vars)

유의한 변수(절편 포함): ['const', 'browsing_time', 'past_purchases', 'email_open_count', 'discount_offered']


`model.pvalues < 0.05`로 p-value가 0.05보다 작은 변수만 골라내는 방식은 지금까지 계속 사용해온 것과 동일합니다. 셀 2에서 눈으로 확인했던 대로, age를 제외한 나머지 변수와 절편(const)이 조건을 만족해 5개(절편 포함)가 남았습니다.

**셀 4) 유의한 변수만으로 재적합**

In [11]:
X2 = X[sig_vars]
model2 = sm.Logit(y, X2).fit()
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.597711
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:                  900
Model:                          Logit   Df Residuals:                      895
Method:                           MLE   Df Model:                            4
Date:                Thu, 09 Jul 2026   Pseudo R-squ.:                  0.1007
Time:                        15:57:09   Log-Likelihood:                -537.94
converged:                       True   LL-Null:                       -598.14
Covariance Type:            nonrobust   LLR p-value:                 4.368e-25
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -2.7114      0.255    -10.638      0.000      -3.211      -2.212
browsing_ti

age를 제거하고 재적합한 결과, 남은 네 변수 모두 p-value가 뚜렷하게 작아 유의합니다.

**셀 5) 유의 계수의 합 (문제 2-1 정답)**

In [12]:
coef_sum = round(model2.params.sum(), 3)
print("2-1 정답:", coef_sum)

2-1 정답: -2.197


`model2.params`(절편 포함 4개 계수 + 상수항)를 모두 더했습니다. 문제에서 "절편 포함"이라고 명시했으므로 const 계수(-2.7114)도 그대로 합산에 포함시켰습니다.

**셀 6) discount_offered 5단위 증가 시 오즈비 (문제 2-2 정답)**

In [13]:
import numpy as np

odds_ratio_5 = np.exp(model2.params['discount_offered'] * 5)
print("2-2 정답:", round(odds_ratio_5, 3))

2-2 정답: 1.305


지금까지 구했던 오즈비는 모두 "변수가 1단위 증가할 때"를 기준으로 했지만, 이번에는 "5단위(%p) 증가할 때"를 물었습니다. 로지스틱 회귀에서 오즈비는 `np.exp(계수)`로 구하는데, 이는 정확히는 `np.exp(계수 × 증가폭)`에서 증가폭이 1인 특수한 경우였습니다. 계수 자체가 "변수가 1단위 증가할 때 로그 오즈가 얼마나 증가하는가"를 의미하므로, 변수가 5단위 증가한다면 로그 오즈는 계수의 5배만큼 증가합니다. 그래서 5단위 증가에 대한 오즈비를 구하려면 계수에 5를 곱한 뒤 지수를 취해야 하며, `np.exp(계수)`를 그대로 5제곱하는 것과는 다른 계산이라는 점에 주의해야 합니다. (참고로 `np.exp(계수 × 5)`는 `np.exp(계수) ** 5`와 수학적으로 동일한 값이 되므로, 어느 방식으로 계산해도 결과는 같습니다.)

이 값이 약 1.305라는 것은, 다른 조건이 모두 같을 때 할인율이 5%p 더 높은 고객은 구매 전환 오즈가 약 1.305배 더 높다는 뜻입니다.

> 💡 "n단위 증가 시 오즈비"를 구하는 문제는 `np.exp(계수 * n)`이라는 공식 하나로 기억해두면, 1단위든 5단위든 10단위든 동일한 방식으로 대응할 수 있습니다. n=1을 대입하면 지금까지 써온 기본 오즈비 공식과 정확히 같아집니다.

**셀 7) 전체 데이터의 구매 전환 확률 예측**

In [14]:
pred_prob = model2.predict(X2)
print(pred_prob.head())

0    0.794681
1    0.434669
2    0.320543
3    0.323718
4    0.423815
dtype: float64


이 문제는 train/test로 나누지 않고 전체 데이터에 그대로 모형을 적용해 확률을 예측했습니다. 지금까지의 문제들처럼 새로운 미래 데이터에 대한 일반화 성능을 확인하려는 목적이 아니라, "지금 가지고 있는 고객 전체 중 몇 명이 조건에 해당하는지"를 세는 것이 이 문제의 목적이기 때문입니다.

**셀 8) 확률 0.3 초과 고객 수 (문제 2-3 정답)**

In [15]:
count = (pred_prob > 0.3).sum()
print("2-3 정답:", count)

2-3 정답: 555


`pred_prob > 0.3`으로 각 고객의 예측 확률이 0.3을 초과하는지 True/False로 판별한 뒤, `.sum()`으로 True인 고객의 수를 세었습니다. 전체 900명 중 555명(약 61.7%)이 이 기준을 넘는 것으로 나타났습니다. 이는 실제 데이터의 전환 비율(약 38.1%)보다 꽤 높은 수치인데, 그 이유는 임계값 0.3이 실제 전환 여부를 가르는 기준(보통 0.5)보다 낮게 설정되어 있어서, "전환 가능성이 어느 정도라도 있는" 고객까지 폭넓게 포함하기 때문입니다.

### 전체 코드

In [16]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv('promotion_conversion.csv')

X = df.drop(columns=['id', 'converted'])
y = df['converted']
X = sm.add_constant(X)

# 전체 변수로 적합
model = sm.Logit(y, X).fit()

# 유의한 변수만 재적합
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
X2 = X[sig_vars]
model2 = sm.Logit(y, X2).fit()

# 2-1: 유의 계수 합(절편 포함)
coef_sum = round(model2.params.sum(), 3)
print("2-1 정답:", coef_sum)

# 2-2: discount_offered 5단위 증가 시 오즈비
odds_ratio_5 = round(np.exp(model2.params['discount_offered'] * 5), 3)
print("2-2 정답:", odds_ratio_5)

# 2-3: 확률 0.3 초과 고객 수
pred_prob = model2.predict(X2)
count = (pred_prob > 0.3).sum()
print("2-3 정답:", count)

Optimization terminated successfully.
         Current function value: 0.597497
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.597711
         Iterations 5
2-1 정답: -2.197
2-2 정답: 1.305
2-3 정답: 555


### 정리

이 문제는 로지스틱 회귀에서 배운 여러 개념(유의변수 재적합, 계수 합, 오즈비, 확률 기반 인원수 세기)을 한 문제 안에 모아서 종합적으로 연습했습니다. 특히 "n단위 증가 시 오즈비"는 계수에 증가폭을 곱한 뒤 지수를 취한다는 확장된 공식을 새로 다뤘는데, 이 공식만 정확히 기억해두면 1단위든 몇 단위든 같은 방식으로 풀 수 있습니다. 개인적으로는, 3유형에서 마주치는 오즈비 문제들은 결국 "계수 × 증가폭에 지수를 취한다"는 하나의 원리에서 파생된 변형일 뿐이므로, 매번 새로운 공식을 외우기보다 이 원리 자체를 이해해두면 어떤 형태로 문제가 바뀌어도 대응할 수 있습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 같은 대상을 두 번 측정한 데이터 비교 | `ttest_rel(a, b)` | 대응표본 t검정, 인자 순서(a-b)가 곧 차이의 방향 |
| 차이값의 평균과 산포를 함께 고려한 지표 | t통계량 = 차이의 평균 / 표준오차 | 평균 차이가 커도 개인차가 크면 t통계량은 작아질 수 있음 |
| 변수가 n단위 증가할 때의 오즈비 | `np.exp(계수 * n)` | 계수는 1단위 증가당 로그 오즈 증가량이므로 n배 필요 |
| 예측 확률 기준으로 인원수 세기 | `(pred_prob > 임계값).sum()` | True/False가 1/0으로 취급되어 합계가 개수가 됨 |
| train/test 분할 없이 전체 데이터에 적용 | 목적에 따라 분할 여부가 달라짐 | 일반화 성능 확인이 아니라 현재 데이터의 현황 파악이 목적일 때는 분할 불필요 |